# Pipeline 2 - Consultas Avançadas

Notebook interativo para demonstrar consultas complexas com SQLAlchemy ORM, usando JOINs, agregações, subconsultas, eager loading e transações.

Execute as células abaixo na ordem desejada. Se necessário, rode primeiro `python3 popular_bd.py` para garantir que o banco tenha os dados de exemplo.

In [1]:
# Setup: importe a app e os modelos (execute uma vez)
from app import create_app
from models import db, Artista, Usuario, Musica, Playlist, MusicaPlaylist
from sqlalchemy import and_, desc, func
from sqlalchemy.orm import aliased, joinedload

app = create_app()
with app.app_context():
    db.create_all()

print('App carregada e tabelas conferidas')

App carregada e tabelas conferidas


# 2.1 Consultas Focadas em Relacionamentos

Nesta seção estão as consultas de navegação entre entidades e filtros em múltiplos relacionamentos.

In [2]:
def listar_playlists_usuario(username):
    with app.app_context():
        return (
            db.session.query(Playlist.nome, Playlist.data_criacao)
            .join(Usuario, Playlist.usuario_id == Usuario.id)
            .filter(Usuario.username == username)
            .order_by(Playlist.data_criacao.asc(), Playlist.nome.asc())
            .all()
        )

def exibir_playlists_usuario(username='Pablo'):
    resultados = listar_playlists_usuario(username)
    print(f'Playlists de {username}:')
    if not resultados:
        print('  Nenhuma playlist encontrada.')
        return resultados
    for nome, data_criacao in resultados:
        print(f'- {nome} | criada em: {data_criacao}')
    return resultados

exibir_playlists_usuario('Pablo')

Playlists de Pablo:
  Nenhuma playlist encontrada.


[]

In [3]:
def listar_musicas_em_playlists_de_usuario_com_artista(username='Josue', artista_nome='Queen'):
    with app.app_context():
        return (
            db.session.query(Playlist.nome.label('playlist'), Musica.titulo, Artista.nome.label('artista'))
            .join(Usuario, Playlist.usuario_id == Usuario.id)
            .join(MusicaPlaylist, and_(Playlist.playlist_id == MusicaPlaylist.playlist_id, Playlist.usuario_id == MusicaPlaylist.usuario_id))
            .join(Musica, Musica.id == MusicaPlaylist.musica_id)
            .join(Artista, Artista.id == Musica.artista_id)
            .filter(Usuario.username == username, Artista.nome == artista_nome)
            .order_by(Playlist.nome.asc(), Musica.titulo.asc())
            .distinct()
            .all()
        )

def exibir_musicas_em_playlists_de_usuario_com_artista(username='Josue', artista_nome='Queen'):
    resultados = listar_musicas_em_playlists_de_usuario_com_artista(username, artista_nome)
    print(f'Músicas de {artista_nome} em playlists de {username}:')
    if not resultados:
        print('  Nenhum registro encontrado.')
        return resultados
    for playlist, titulo, artista in resultados:
        print(f'- {titulo} | playlist: {playlist} | artista: {artista}')
    return resultados

exibir_musicas_em_playlists_de_usuario_com_artista('Josue', 'Queen')

Músicas de Queen em playlists de Josue:
  Nenhum registro encontrado.


[]

In [4]:
def contar_musicas_por_playlist():
    with app.app_context():
        total_musicas = func.count(MusicaPlaylist.musica_id).label('total_musicas')
        return (
            db.session.query(Playlist.nome, Usuario.username, total_musicas)
            .join(Usuario, Playlist.usuario_id == Usuario.id)
            .outerjoin(MusicaPlaylist, and_(Playlist.playlist_id == MusicaPlaylist.playlist_id, Playlist.usuario_id == MusicaPlaylist.usuario_id))
            .group_by(Playlist.playlist_id, Playlist.usuario_id, Playlist.nome, Usuario.username)
            .order_by(desc(total_musicas), Playlist.nome.asc())
            .all()
        )

def exibir_contagem_musicas_por_playlist():
    resultados = contar_musicas_por_playlist()
    print('Contagem de músicas por playlist:')
    for nome, username, total_musicas in resultados:
        print(f'- {nome} | dono: {username} | total: {total_musicas}')
    return resultados

exibir_contagem_musicas_por_playlist()

Contagem de músicas por playlist:


[]

In [5]:
def listar_artistas_sem_musicas_em_playlists():
    with app.app_context():
        return (
            db.session.query(Artista.nome)
            .outerjoin(Musica, Artista.id == Musica.artista_id)
            .outerjoin(MusicaPlaylist, Musica.id == MusicaPlaylist.musica_id)
            .group_by(Artista.id, Artista.nome)
            .having(func.count(MusicaPlaylist.musica_id) == 0)
            .order_by(Artista.nome.asc())
            .all()
        )

def exibir_artistas_sem_musicas_em_playlists():
    resultados = listar_artistas_sem_musicas_em_playlists()
    print('Artistas sem músicas adicionadas a playlists:')
    if not resultados:
        print('  Nenhum artista encontrado.')
        return resultados
    for (nome,) in resultados:
        print(f'- {nome}')
    return resultados

exibir_artistas_sem_musicas_em_playlists()

Artistas sem músicas adicionadas a playlists:
  Nenhum artista encontrado.


[]

# 2.2 Consultas Focadas em Otimização e Detalhe

Agora a ênfase é em carregamento otimizado, agregações e subconsultas.

In [6]:
def buscar_musica_com_artista(musica_id):
    with app.app_context():
        return (
            db.session.query(Musica)
            .options(joinedload(Musica.artista))
            .filter(Musica.id == musica_id)
            .first()
        )

def exibir_musica_com_artista(musica_id):
    musica = buscar_musica_com_artista(musica_id)
    print(f'Detalhes da música {musica_id}:')
    if not musica:
        print('  Música não encontrada.')
        return None
    print(f'- Título: {musica.titulo}')
    print(f'- Duração: {musica.duracao_segundos} segundos')
    print(f'- Artista: {musica.artista.nome}')
    return musica

exibir_musica_com_artista(1)

Detalhes da música 1:
  Música não encontrada.


In [7]:
def formatar_segundos(total_segundos):
    total_segundos = int(total_segundos or 0)
    horas, resto = divmod(total_segundos, 3600)
    minutos, segundos = divmod(resto, 60)
    return f'{horas:02d}:{minutos:02d}:{segundos:02d}'

def calcular_tempo_total_por_playlist():
    with app.app_context():
        tempo_total = func.coalesce(func.sum(Musica.duracao_segundos), 0).label('tempo_total_segundos')
        return (
            db.session.query(Playlist.nome, Usuario.username, tempo_total)
            .join(Usuario, Playlist.usuario_id == Usuario.id)
            .outerjoin(MusicaPlaylist, and_(Playlist.playlist_id == MusicaPlaylist.playlist_id, Playlist.usuario_id == MusicaPlaylist.usuario_id))
            .outerjoin(Musica, Musica.id == MusicaPlaylist.musica_id)
            .group_by(Playlist.playlist_id, Playlist.usuario_id, Playlist.nome, Usuario.username)
            .order_by(Usuario.username.asc(), Playlist.nome.asc())
            .all()
        )

def exibir_tempo_total_por_playlist():
    resultados = calcular_tempo_total_por_playlist()
    print('Tempo total de reprodução por playlist:')
    for nome, username, total_segundos in resultados:
        print(f'- {nome} | dono: {username} | total: {total_segundos} s | {formatar_segundos(total_segundos)}')
    return resultados

exibir_tempo_total_por_playlist()

Tempo total de reprodução por playlist:


[]

In [8]:
def listar_musicas_mais_curta_que_media_do_artista():
    with app.app_context():
        musica_media = aliased(Musica)
        media_artista = (
            db.session.query(func.avg(musica_media.duracao_segundos))
            .filter(musica_media.artista_id == Musica.artista_id)
            .correlate(Musica)
            .scalar_subquery()
        )
        return (
            db.session.query(Musica.titulo, Musica.duracao_segundos, Artista.nome.label('artista'))
            .join(Artista, Artista.id == Musica.artista_id)
            .filter(Musica.duracao_segundos < media_artista)
            .order_by(Artista.nome.asc(), Musica.duracao_segundos.asc())
            .all()
        )

def exibir_musicas_mais_curta_que_media_do_artista():
    resultados = listar_musicas_mais_curta_que_media_do_artista()
    print('Músicas mais curtas que a média do próprio artista:')
    if not resultados:
        print('  Nenhuma música encontrada.')
        return resultados
    for titulo, duracao, artista in resultados:
        print(f'- {titulo} | artista: {artista} | duração: {duracao} s')
    return resultados

exibir_musicas_mais_curta_que_media_do_artista()

Músicas mais curtas que a média do próprio artista:
  Nenhuma música encontrada.


[]

# 2.3 Chaves e Junções Complexas

Aqui entram consultas sobre a tabela de junção com atributos extras e navegação inversa pela chave composta.

In [9]:
def listar_musicas_da_playlist_com_ordem(nome_playlist='Rock do Pablo', username='Pablo'):
    with app.app_context():
        return (
            db.session.query(Musica.titulo, MusicaPlaylist.ordem_na_playlist)
            .join(MusicaPlaylist, Musica.id == MusicaPlaylist.musica_id)
            .join(Playlist, and_(Playlist.playlist_id == MusicaPlaylist.playlist_id, Playlist.usuario_id == MusicaPlaylist.usuario_id))
            .join(Usuario, Usuario.id == Playlist.usuario_id)
            .filter(Playlist.nome == nome_playlist, Usuario.username == username)
            .order_by(MusicaPlaylist.ordem_na_playlist.asc())
            .all()
        )

def exibir_musicas_da_playlist_com_ordem(nome_playlist='Rock do Pablo', username='Pablo'):
    resultados = listar_musicas_da_playlist_com_ordem(nome_playlist, username)
    print(f'Músicas na playlist {nome_playlist}:')
    if not resultados:
        print('  Nenhuma música encontrada.')
        return resultados
    for titulo, ordem in resultados:
        print(f'- ordem {ordem}: {titulo}')
    return resultados

exibir_musicas_da_playlist_com_ordem('Rock do Pablo', 'Pablo')

Músicas na playlist Rock do Pablo:
  Nenhuma música encontrada.


[]

In [10]:
def buscar_usuario_dono_da_playlist_que_contem_musica(titulo_musica='Bohemian Rhapsody'):
    with app.app_context():
        return (
            db.session.query(Usuario.username)
            .select_from(Musica)
            .join(MusicaPlaylist, Musica.id == MusicaPlaylist.musica_id)
            .join(Playlist, and_(Playlist.playlist_id == MusicaPlaylist.playlist_id, Playlist.usuario_id == MusicaPlaylist.usuario_id))
            .join(Usuario, Usuario.id == Playlist.usuario_id)
            .filter(Musica.titulo == titulo_musica)
            .distinct()
            .all()
        )

def exibir_usuario_dono_da_playlist_que_contem_musica(titulo_musica='Bohemian Rhapsody'):
    resultados = buscar_usuario_dono_da_playlist_que_contem_musica(titulo_musica)
    print(f'Usuário(s) dono(s) da playlist que contém {titulo_musica}:')
    if not resultados:
        print('  Nenhum usuário encontrado.')
        return resultados
    for (username,) in resultados:
        print(f'- {username}')
    return resultados

exibir_usuario_dono_da_playlist_que_contem_musica('Bohemian Rhapsody')

Usuário(s) dono(s) da playlist que contém Bohemian Rhapsody:
  Nenhum usuário encontrado.


[]

# 2.4 Funções e Subconsultas

Esta etapa demonstra ranking com window functions e comparação com subconsulta escalar.

In [11]:
def rank_artistas_por_playlists():
    with app.app_context():
        presencas = (
            db.session.query(
                Artista.id.label('artista_id'),
                Artista.nome.label('artista_nome'),
                Playlist.playlist_id.label('playlist_id'),
                Playlist.usuario_id.label('usuario_id'),
            )
            .join(Musica, Artista.id == Musica.artista_id)
            .join(MusicaPlaylist, Musica.id == MusicaPlaylist.musica_id)
            .join(Playlist, and_(Playlist.playlist_id == MusicaPlaylist.playlist_id, Playlist.usuario_id == MusicaPlaylist.usuario_id))
            .distinct()
            .subquery()
        )

        totais = (
            db.session.query(
                presencas.c.artista_id,
                presencas.c.artista_nome,
                func.count().label('total_playlists'),
            )
            .group_by(presencas.c.artista_id, presencas.c.artista_nome)
            .subquery()
        )

        ranking = func.dense_rank().over(order_by=totais.c.total_playlists.desc()).label('ranking')

        return (
            db.session.query(totais.c.artista_nome, totais.c.total_playlists, ranking)
            .order_by(ranking.asc(), totais.c.artista_nome.asc())
            .all()
        )

def exibir_rank_artistas_por_playlists():
    resultados = rank_artistas_por_playlists()
    print('Ranking de artistas por quantidade de playlists em que aparecem:')
    for artista, total_playlists, ranking in resultados:
        print(f'- {ranking}º | {artista} | playlists: {total_playlists}')
    return resultados

def listar_musicas_led_zeppelin_maiores_que_a_maior_musica_de_queen():
    with app.app_context():
        maior_queen = (
            db.session.query(func.max(Musica.duracao_segundos))
            .join(Artista, Artista.id == Musica.artista_id)
            .filter(Artista.nome == 'Queen')
            .scalar_subquery()
        )
        return (
            db.session.query(Musica.titulo, Musica.duracao_segundos)
            .join(Artista, Artista.id == Musica.artista_id)
            .filter(Artista.nome == 'Led Zeppelin', Musica.duracao_segundos > maior_queen)
            .order_by(Musica.duracao_segundos.desc())
            .all()
        )

def exibir_musicas_led_zeppelin_maiores_que_a_maior_musica_de_queen():
    resultados = listar_musicas_led_zeppelin_maiores_que_a_maior_musica_de_queen()
    print('Músicas de Led Zeppelin maiores que a maior música de Queen:')
    if not resultados:
        print('  Nenhuma música encontrada.')
        return resultados
    for titulo, duracao in resultados:
        print(f'- {titulo} | duração: {duracao} s')
    return resultados

exibir_rank_artistas_por_playlists()
print()
exibir_musicas_led_zeppelin_maiores_que_a_maior_musica_de_queen()

Ranking de artistas por quantidade de playlists em que aparecem:

Músicas de Led Zeppelin maiores que a maior música de Queen:
  Nenhuma música encontrada.


[]

# 2.5 Teste de Transações e Concorrência

A função abaixo faz a transferência de uma música entre playlists do mesmo usuário de forma atômica.

In [12]:
def transferir_musica_entre_playlists(usuario_id, playlist_origem_id, playlist_destino_id, musica_id):
    with app.app_context():
        if playlist_origem_id == playlist_destino_id:
            raise ValueError('A origem e o destino devem ser playlists diferentes')

        origem = Playlist.query.filter_by(playlist_id=playlist_origem_id, usuario_id=usuario_id).first()
        destino = Playlist.query.filter_by(playlist_id=playlist_destino_id, usuario_id=usuario_id).first()

        if not origem or not destino:
            raise ValueError('As playlists precisam existir e pertencer ao mesmo usuário')

        mp_origem = MusicaPlaylist.query.filter_by(musica_id=musica_id, playlist_id=playlist_origem_id, usuario_id=usuario_id).first()
        if not mp_origem:
            raise ValueError('A música não está na playlist de origem')

        if MusicaPlaylist.query.filter_by(musica_id=musica_id, playlist_id=playlist_destino_id, usuario_id=usuario_id).first():
            raise ValueError('A música já está na playlist de destino')

        try:
            db.session.delete(mp_origem)
            ultima_ordem = (
                db.session.query(func.max(MusicaPlaylist.ordem_na_playlist))
                .filter_by(playlist_id=playlist_destino_id, usuario_id=usuario_id)
                .scalar()
            )
            nova_ordem = (ultima_ordem or 0) + 1
            nova_relacao = MusicaPlaylist(
                musica_id=musica_id,
                playlist_id=playlist_destino_id,
                usuario_id=usuario_id,
                ordem_na_playlist=nova_ordem,
            )
            db.session.add(nova_relacao)
            db.session.commit()
            return nova_relacao
        except Exception:
            db.session.rollback()
            raise

def exibir_transferencia_exemplo():
    with app.app_context():
        pablo = Usuario.query.filter_by(username='Pablo').first()
        if not pablo:
            print('Usuário Pablo não encontrado.')
            return None

        origem = Playlist.query.filter_by(nome='Heavy Riffs', usuario_id=pablo.id).first()
        destino = Playlist.query.filter_by(nome='Rock do Pablo', usuario_id=pablo.id).first()
        musica = Musica.query.filter_by(titulo='Thunderstruck').first()

        if not origem or not destino or not musica:
            print('Não foi possível montar o exemplo de transferência.')
            return None

        presente_na_origem = MusicaPlaylist.query.filter_by(
            musica_id=musica.id,
            playlist_id=origem.playlist_id,
            usuario_id=pablo.id
        ).first()

        if not presente_na_origem:
            print('A música já foi transferida anteriormente ou não está mais na playlist de origem.')
            return None

        if MusicaPlaylist.query.filter_by(musica_id=musica.id, playlist_id=destino.playlist_id, usuario_id=pablo.id).first():
            print('A música já está na playlist de destino.')
            return None

        resultado = transferir_musica_entre_playlists(pablo.id, origem.playlist_id, destino.playlist_id, musica.id)
        print('Transferência concluída com sucesso:')
        print(f'- música: {musica.titulo}')
        print(f'- origem: {origem.nome}')
        print(f'- destino: {destino.nome}')
        print(f'- nova ordem na destino: {resultado.ordem_na_playlist}')
        return resultado

exibir_transferencia_exemplo()

Usuário Pablo não encontrado.
